In [2]:
import pandas as pd
from pathlib import Path

In [3]:
db_path = Path(r"E:\BDDPlabacomCoordinador")
to_save_path = Path(r"E:\ProyectoAnalisisElectrico\MedidasValorizadas")

In [4]:
dst_changes = {
    "2604": {"day": 4, "jump": -1, "instant": "24:00"},
    "2509": {"day": 6, "jump": 1,  "instant": "24:00"}
}

In [5]:
measurements_list = [
    "Medidas_Valorizadas_15min_Norte Distribución",
    "Medidas_Valorizadas_15min_Norte",
    "Medidas_Valorizadas_15min_Sur Distribución",
    "Medidas_Valorizadas_15min_Sur"
]

In [6]:
agg_rules = {
    'medida_3': ['sum', 'min'], 
    'CMg[CLP/KWh]': 'mean', 
    'valorizado_CLP': 'sum',
    'Fecha_Medicion': 'last'
}

group_columns = [
    'Hora', 'clave', 'nombre_barra', 'tension', 
    'Zona', 'Razon_Social', 'RUT', 'Nombre_Corto', 'tipo'
]

In [10]:
pth = Path(r"D:\BDDPlabacomCoordinador\PLABACOM_2603_BD01\02 Medidas por tipo") 
df1 = pd.read_csv(pth/"Medidas_Valorizadas_15min_Norte"/"Medidas_Valorizadas_15min_Norte.csv", sep=";")
df2 = pd.read_csv(pth/"Medidas_Valorizadas_15min_Norte Distribución"/"Medidas_Valorizadas_15min_Norte Distribución.csv", sep=";")
df3 = pd.read_csv(pth/"Medidas_Valorizadas_15min_Sur"/"Medidas_Valorizadas_15min_Sur.csv", sep=";")
df4 = pd.read_csv(pth/"Medidas_Valorizadas_15min_Sur Distribución"/"Medidas_Valorizadas_15min_Sur Distribución.csv", sep=";")

C:\Users\joser\AppData\Local\Temp\ipykernel_18428\727786256.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(pth/"Medidas_Valorizadas_15min_Norte Distribución"/"Medidas_Valorizadas_15min_Norte Distribución.csv", sep=";")


In [11]:
df=pd.concat([df1,df2,df3,df4])

In [12]:
df = df[pd.to_numeric(df["RUT"].str.split("-").str[0].str.replace(".", "", regex=False), errors="coerce") >= 50000000] 

In [14]:
df = df[(df["tipo"] == "L") | (df["tipo"] == "L_D")]

In [15]:
len(df["RUT"].unique())

72

In [ ]:
for folder_date in db_path.iterdir():
    if not folder_date.is_dir():
        continue
        
    origin, date, version = folder_date.name.split("_")
    
    if int(date) < 2505: 
        continue

    dst_info = dst_changes.get(date)
    to_save_folder = to_save_path / f"{date}"
    
    if to_save_folder.is_dir():
        print(f"Data for {date} already processed. Skipping...")
        continue 
        
    to_save_folder.mkdir(parents=True, exist_ok=True)
    measurements_folder = folder_date / "02 Medidas por tipo"

    dfs = []
    dfs_bad = []

    print(f"Starting {date}...")

    for measurement_name in measurements_list:
        csv_path = measurements_folder / measurement_name / "{}.csv".format(measurement_name)

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path, sep=";", dtype={"clave": str})  

        df = df[(df["tipo"] == "L") | (df["tipo"] == "L_D")]
        df = df[pd.to_numeric(df["RUT"].str.split("-").str[0].str.replace(".", "", regex=False), errors="coerce") >= 50000000] 
        
        df["Fecha_Medicion"] = pd.to_datetime(df["Fecha_Medicion"], format="%Y-%m-%d %H:%M:%S")
        df["Hora"] = (df["Fecha_Medicion"].dt.day-1)*24 + df["Fecha_Medicion"].dt.hour 

        groups = df.groupby(by=group_columns, sort=False) 
        groups_with_size = groups.size()
        
        allowed_repeated_hour = -1 
        if dst_info and dst_info["jump"] == -1:
            allowed_repeated_hour = (dst_info["day"] - 1) * 24 + 23
            
        group_hours = groups_with_size.index.get_level_values('Hora')
        
        mask_4 = (groups_with_size == 4)
        valid_mask_8 = (groups_with_size == 8) & (group_hours == allowed_repeated_hour)
        
        good_groups_mask = mask_4 | valid_mask_8
        df_aggregated = groups.agg(agg_rules)
        
        if valid_mask_8.any():
            df_aggregated.loc[valid_mask_8, ('medida_3', 'sum')] /= 2 
            df_aggregated.loc[valid_mask_8, ('valorizado_CLP', 'sum')] /= 2

        df_valid = df_aggregated[good_groups_mask].reset_index()
        df_valid.columns = [f"{col[0]}_{col[1]}" if isinstance(col, tuple) and col[1] else col[0] for col in df_valid.columns]
        
        if dst_info and dst_info["jump"] == 1:
            missing_hour = dst_info["day"] * 24
            
            if missing_hour not in df_valid['Hora'].values:
                prev_hour = missing_hour - 1
                next_hour = missing_hour + 1
                
                df_prev = df_valid[df_valid['Hora'] == prev_hour].set_index('clave')
                df_next = df_valid[df_valid['Hora'] == next_hour].set_index('clave')
                
                common_keys = df_prev.index.intersection(df_next.index)
                
                if not common_keys.empty:
                    df_missing = pd.DataFrame(index=common_keys)
                    df_missing['Hora'] = missing_hour
                    
                    num_cols = [c for c in df_valid.columns if c.endswith('_sum') or c.endswith('_mean') or c.endswith('_min')]
                    cat_cols = [c for c in df_valid.columns if c not in num_cols and c != 'Hora' and c != 'clave']
                    
                    # Fix: Explicitly handle the timestamp shift for the injected hour
                    for c in cat_cols:
                        if c == 'Fecha_Medicion_last':
                            df_missing[c] = df_prev.loc[common_keys, c] + pd.Timedelta(hours=1)
                        else:
                            df_missing[c] = df_prev.loc[common_keys, c]
                        
                    for c in num_cols:
                        df_missing[c] = (df_prev.loc[common_keys, c] + df_next.loc[common_keys, c]) / 2.0
                        
                    df_missing = df_missing.reset_index(names='clave')
                    
                    df_valid = pd.concat([df_valid, df_missing], ignore_index=True)
                    df_valid = df_valid.sort_values(by=['clave', 'Hora']).reset_index(drop=True)

        dfs.append(df_valid)
        
        bad_groups_mask = ~good_groups_mask
        if bad_groups_mask.any(): 
            print(f"Issues detected in: {measurement_name}")
            
            def is_invalid_group(x):
                sz = len(x)
                if sz == 4:
                    return False
                if sz == 8 and dst_info and dst_info["jump"] == -1:
                    if x['Hora'].iloc[0] == allowed_repeated_hour:
                        return False
                return True

            bad_df = groups.filter(is_invalid_group)
            dfs_bad.append(bad_df)
            
    if dfs:
        final_valid_df = pd.concat(dfs, ignore_index=True)
        final_valid_df.to_parquet(to_save_folder / f"{date}_medidas_horarias.parquet", engine="pyarrow", compression="snappy")
        
    if dfs_bad:
        final_bad_df = pd.concat(dfs_bad, ignore_index=True)
        final_bad_df.to_csv(to_save_folder / f"{date}_auditoria_errores_15min.csv", sep=";", index=False, encoding="utf-8")

print("\nProcess Finished.")

Starting 2509...

Process Finished.
